In [2]:
import pandas as pd
import numpy as np
import os
import gc
from scipy.sparse import csr_matrix

ruta_raw = 'data/raw'

df_books = pd.read_csv(os.path.join(ruta_raw, 'books.csv'), sep=',', encoding='latin-1', on_bad_lines='skip', low_memory=False)
df_copies = pd.read_csv(os.path.join(ruta_raw, 'copies(ejemplares).csv'), sep=',', encoding='latin-1')
df_ratings = pd.read_csv(os.path.join(ruta_raw, 'ratings.csv'), sep=',', encoding='latin-1')

In [3]:
df_val_libros = pd.merge(df_ratings, df_copies, on='copy_id')
df_completo = pd.merge(df_val_libros, df_books[['book_id', 'title']], on='book_id')

top_valorados = df_completo.groupby('title')['rating'].count().sort_values(ascending=False).head(10)

print("Estadísticas iniciales:")
print(f"Total de valoraciones procesadas: {len(df_completo)}")
print("\nTop 10 Libros más valorados (Insumo CU-04):")
print(top_valorados)

Estadísticas iniciales:
Total de valoraciones procesadas: 5976198

Top 10 Libros más valorados (Insumo CU-04):
title
The Hunger Games (The Hunger Games, #1)                        22806
Harry Potter and the Sorcerer''s Stone (Harry Potter, #1)      21850
To Kill a Mockingbird                                          19088
Twilight (Twilight, #1)                                        16931
The Great Gatsby                                               16604
Catching Fire (The Hunger Games, #2)                           16549
Mockingjay (The Hunger Games, #3)                              15953
Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)    15855
Harry Potter and the Chamber of Secrets (Harry Potter, #2)     15657
The Hobbit                                                     15558
Name: rating, dtype: int64


In [4]:
conteo_libros = df_completo.groupby('book_id')['rating'].count()
conteo_usuarios = df_completo.groupby('user_id')['rating'].count()

libros_top = conteo_libros[conteo_libros >= 1000].index
usuarios_top = conteo_usuarios[conteo_usuarios >= 100].index

df_reco = df_completo[(df_completo['book_id'].isin(libros_top)) & (df_completo['user_id'].isin(usuarios_top))].copy()

df_reco['rating'] = df_reco['rating'].astype(np.int8)
df_reco['user_id'] = df_reco['user_id'].astype(np.int32)
df_reco['book_id'] = df_reco['book_id'].astype(np.int32)

del df_ratings
del df_copies
del df_val_libros
gc.collect()

print(f"Registros finales tras filtrado de densidad: {len(df_reco)}")

Registros finales tras filtrado de densidad: 2748474


In [5]:
usuarios = sorted(df_reco['user_id'].unique())
libros = sorted(df_reco['book_id'].unique())

mapa_usuarios = {id: i for i, id in enumerate(usuarios)}
mapa_libros = {id: i for i, id in enumerate(libros)}

filas = df_reco['user_id'].map(mapa_usuarios)
columnas = df_reco['book_id'].map(mapa_libros)

matriz_sparse = csr_matrix((np.ones(len(df_reco), dtype=np.int8), (filas, columnas)), shape=(len(usuarios), len(libros)))

print("Dimensiones de la matriz de recomendación (Usuarios x Libros):")
print(matriz_sparse.shape)

Dimensiones de la matriz de recomendación (Usuarios x Libros):
(37082, 1229)
